## Out of distribution test

This is the last experiment in the thesis. It asks whether MT-TrajNet still works on a product code it has never seen, and whether any loss on the high hardness regime is a regime shift effect or just the ordinary cost of predicting an unseen product.

Two codes are tested. Code 15 is the high hardness case, 64 batches, held out of the cross validation entirely so no fold model has ever seen it. Code 25 is the low cluster control, 34 batches, and it sits in fold 1's test set so the fold 1 model never trained on it either. The fold 1 model is the only model that has never seen either code, so it is used for both and the contrast between them is fair.

Two folders are used and they stay separate. DATA is the raw dataset folder and is only read from. RESULTS is the results folder inside the repository and is the only place this notebook writes to. Both are checked before anything runs, and the data file is checked against the fingerprint of the copy that was verified against the raw sensor CSVs.

Nothing is trained here. The saved fold models are loaded and run in inference mode, so this runs on the laptop with no GPU.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle,json,hashlib
from pathlib import Path

DATA=Path(r"C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets")
REPO=Path(r"C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis")
RESULTS=REPO/"results"

assert DATA.exists(),f"data folder not found: {DATA}"
assert RESULTS.exists(),f"results folder not found: {RESULTS}"
need=["mttrajnet_fold1.pt","mttrajnet_fold2.pt","mttrajnet_fold3.pt","mttrajnet_stratified.json"]
for f in need:
    assert (RESULTS/f).exists(),f"missing: {RESULTS/f}"

print("reading data from :",DATA)
print("reading models from:",RESULTS)
print()

h=hashlib.md5()
with open(DATA/"assembled_trajectories.pkl","rb") as fh:
    for c in iter(lambda:fh.read(1<<20),b""): h.update(c)
print("pkl md5:",h.hexdigest())
print("pkl is the verified copy:",h.hexdigest()=="7d7fc76be1e4940198b76d9d0797a3a9")

with open(DATA/"assembled_trajectories.pkl","rb") as fh:
    data=pickle.load(fh)
X_traj=data["X_traj"]
y_arr=data["y_arr"]
groups=data["groups"]

targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]
OOD_CODE=15
CONTROL_CODE=25

print()
print("batches:",len(X_traj),"| codes:",len(np.unique(groups)))
print(f"code {OOD_CODE} high cluster OOD : {int((groups==OOD_CODE).sum())} batches, mean hardness {y_arr[groups==OOD_CODE,1].mean():.1f}")
print(f"code {CONTROL_CODE} low cluster control: {int((groups==CONTROL_CODE).sum())} batches, mean hardness {y_arr[groups==CONTROL_CODE,1].mean():.1f}")

reading data from : C:\Users\Arpit Joshua Elias\OneDrive\Desktop\Pharma datasets
reading models from: C:\Users\Arpit Joshua Elias\Projects\mt-trajnet-thesis\results

pkl md5: 7d7fc76be1e4940198b76d9d0797a3a9
pkl is the verified copy: True

batches: 1005 | codes: 25
code 15 high cluster OOD : 64 batches, mean hardness 90.2
code 25 low cluster control: 34 batches, mean hardness 50.4


## Rebuild the model and load the saved weights

The architecture is defined exactly as in Notebook 8 so the saved state dictionaries load without any mismatch. Each saved fold file also carries the normalisation statistics that fold was trained with. This matters, because the input must be normalised with the statistics the model saw during training, not with statistics recomputed here.

Each fold file also stores its training index. That lets me verify directly, rather than assume, that no fold model ever saw code 15, and that the fold 1 model never saw code 25. If either check fails the test is invalid and nothing below should be trusted.

In [2]:
STRIDE=2
MAXLEN=6000

def prep_batch(traj_list,mean,std,maxlen=MAXLEN):
    out=[]
    for a in traj_list:
        a=a[::STRIDE]
        a=(a-mean)/std
        if len(a)<maxlen:
            pad=np.zeros((maxlen-len(a),a.shape[1]),dtype="float32")
            a=np.vstack([a,pad])
        else:
            a=a[:maxlen]
        out.append(a)
    arr=np.array(out,dtype="float32")
    return torch.tensor(arr).transpose(1,2)

class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU()
        self.drop=nn.Dropout(dropout)
        self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None
    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad]
        out=self.drop(self.relu(out))
        return self.relu(out+res)

class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,1,dropout=dropout)
        self.b2=TCNBlock(hidden,hidden,2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,4,dropout=dropout)
        self.b4=TCNBlock(hidden,hidden,8,dropout=dropout)
    def forward(self,x):
        x=self.b1(x);x=self.b2(x);x=self.b3(x);x=self.b4(x)
        return x.mean(dim=2)

class EvidentialHead(nn.Module):
    def __init__(self,in_dim,hidden=64,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,4))
    def forward(self,x):
        out=self.net(x)
        gamma=out[:,0]
        nu=F.softplus(out[:,1]).clamp(min=1e-2,max=1e3)
        alpha=F.softplus(out[:,2]).clamp(min=1e-2,max=1e3)+1.0
        beta=F.softplus(out[:,3]).clamp(min=1e-2,max=1e3)
        return gamma,nu,alpha,beta

class MTTrajNet(nn.Module):
    def __init__(self,in_ch=10,hidden=128,n_targets=4,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.heads=nn.ModuleList([EvidentialHead(hidden,dropout=dropout) for _ in range(n_targets)])
    def forward(self,x):
        z=self.encoder(x)
        outs=[h(z) for h in self.heads]
        gamma=torch.stack([o[0] for o in outs],dim=1)
        nu=torch.stack([o[1] for o in outs],dim=1)
        alpha=torch.stack([o[2] for o in outs],dim=1)
        beta=torch.stack([o[3] for o in outs],dim=1)
        return gamma,nu,alpha,beta

idx15=np.where(groups==OOD_CODE)[0]
idx25=np.where(groups==CONTROL_CODE)[0]

folds={}
for f in [1,2,3]:
    ck=torch.load(RESULTS/f"mttrajnet_fold{f}.pt",map_location="cpu",weights_only=False)
    model=MTTrajNet()
    model.load_state_dict(ck["state"])
    model.eval()
    folds[f]={"model":model,"ck":ck}
    s15=int(np.isin(ck["train_idx"],idx15).sum())
    s25=int(np.isin(ck["train_idx"],idx25).sum())
    print(f"fold {f}: trained on {len(ck['train_idx'])} batches | code 15 seen {s15} | code 25 seen {s25}")

print()
print("check: no fold model saw code 15 in training:",all(int(np.isin(folds[f]["ck"]["train_idx"],idx15).sum())==0 for f in [1,2,3]))
print("check: fold 1 never saw code 25 in training:",int(np.isin(folds[1]["ck"]["train_idx"],idx25).sum())==0)
print("check: all three models loaded:",len(folds)==3)

fold 1: trained on 610 batches | code 15 seen 0 | code 25 seen 0
fold 2: trained on 606 batches | code 15 seen 0 | code 25 seen 34
fold 3: trained on 623 batches | code 15 seen 0 | code 25 seen 34

check: no fold model saw code 15 in training: True
check: fold 1 never saw code 25 in training: True
check: all three models loaded: True


## Predict the two held out codes

The fold 1 model predicts both codes, which keeps the comparison fair since it has seen neither. Code 15 is also predicted by the fold 2 and fold 3 models as a robustness check, because those models have never seen it either, so if the three disagree the result is not stable.

The uncertainty scale is not refitted here. Each fold uses the scale that Notebook 8 fitted for it on the other two folds, so the scale has never touched code 15, code 25, or any prediction reported below. Refitting the scale on the held out code would be exactly the mistake that was corrected in Notebook 8.

In [3]:
cal=json.load(open(RESULTS/"mttrajnet_stratified.json"))["calibration"]
z90=1.6448536269514722

def predict(model,ck,idx,batch_size=8):
    Xb=prep_batch([X_traj[i] for i in idx],ck["xmean"],ck["xstd"])
    gs=[]
    ss=[]
    with torch.no_grad():
        for i in range(0,len(Xb),batch_size):
            g,n,a,b=model(Xb[i:i+batch_size])
            v=b/(n*(a-1))
            gs.append(g.numpy())
            ss.append(torch.sqrt(v).numpy())
    pred=np.vstack(gs)*ck["ystd"]+ck["ymean"]
    std=np.vstack(ss)*ck["ystd"]
    return pred,std

def evaluate(pred,std,idx,fold):
    r={}
    for k in range(4):
        t=y_arr[idx,k]
        p=pred[:,k]
        sc=cal[targets[k]]["scale_per_fold"][fold-1]
        s=std[:,k]*sc
        r[targets[k]]={"rmse":round(float(np.sqrt(np.mean((p-t)**2))),3),
                       "picp":round(float(np.mean((t>=p-z90*s)&(t<=p+z90*s))),3),
                       "mean_pred":round(float(p.mean()),2),
                       "mean_true":round(float(t.mean()),2),
                       "scale_used":sc}
    return r

res={}
for f in [1,2,3]:
    p,s=predict(folds[f]["model"],folds[f]["ck"],idx15)
    res[f"code15_fold{f}"]=evaluate(p,s,idx15,f)
    print(f"code 15 predicted by fold {f} model")

p,s=predict(folds[1]["model"],folds[1]["ck"],idx25)
res["code25_fold1"]=evaluate(p,s,idx25,1)
print("code 25 predicted by fold 1 model")
print()
print("done, 4 prediction runs over",len(idx15)*3+len(idx25),"batch passes")

code 15 predicted by fold 1 model
code 15 predicted by fold 2 model
code 15 predicted by fold 3 model
code 25 predicted by fold 1 model

done, 4 prediction runs over 226 batch passes


## Results

The comparison that matters is the first two rows. The same fold 1 model, on two products it has never seen, one from the high hardness regime and one from the normal regime. The cross validation average is the in distribution reference point.

If code 15 is much worse than code 25, the loss is a regime shift effect and the model struggles specifically with the high hardness regime it barely saw in training. If both are similar, the loss is the ordinary cost of predicting an unseen product and code 15 is not special. The control is what makes those two explanations separable.

In [4]:
cv=json.load(open(RESULTS/"mttrajnet_stratified.json"))["rmse_mean"]

rows=[("code 15 unseen, high cluster","code15_fold1"),
      ("code 25 unseen, control","code25_fold1"),
      ("code 15, fold 2 model","code15_fold2"),
      ("code 15, fold 3 model","code15_fold3")]

print("RMSE, lower is better")
print(f"{'':<32}{'dissolution':<14}{'hardness':<14}{'weight RSD':<14}{'tensile'}")
for label,key in rows:
    v=[res[key][t]["rmse"] for t in targets]
    print(f"{label:<32}{v[0]:<14}{v[1]:<14}{v[2]:<14}{v[3]}")
print(f"{'cross validation average':<32}{cv[targets[0]]:<14}{cv[targets[1]]:<14}{cv[targets[2]]:<14}{cv[targets[3]]}")

print()
print("PICP at 90 percent, 0.90 is the target")
print(f"{'':<32}{'dissolution':<14}{'hardness':<14}{'weight RSD':<14}{'tensile'}")
for label,key in rows:
    v=[res[key][t]["picp"] for t in targets]
    print(f"{label:<32}{v[0]:<14}{v[1]:<14}{v[2]:<14}{v[3]}")

print()
print("hardness, predicted against actual")
for label,key in rows[:2]:
    r=res[key]["tbl_av_hardness"]
    print(f"  {label:<30} predicted {r['mean_pred']:<8} actual {r['mean_true']}")

print()
print("hardness RMSE relative to the cross validation average")
for label,key in rows[:2]:
    print(f"  {label:<30} {res[key]['tbl_av_hardness']['rmse']/cv['tbl_av_hardness']:.2f}x")

out={"analysis":"out of distribution test, code 15 high cluster and code 25 low cluster control",
     "setup":"fold 1 model has never seen either code so the contrast is fair; code 15 also predicted by the fold 2 and fold 3 models as a robustness check; the uncertainty scale is the one Notebook 8 fitted for each fold on the other two folds so it never touched the codes reported here; no training performed, saved weights loaded in inference mode",
     "ood_code":int(OOD_CODE),
     "ood_batches":int(len(idx15)),
     "ood_mean_hardness":round(float(y_arr[idx15,1].mean()),1),
     "control_code":int(CONTROL_CODE),
     "control_batches":int(len(idx25)),
     "control_mean_hardness":round(float(y_arr[idx25,1].mean()),1),
     "cv_reference_rmse":cv,
     "results":res}
with open(RESULTS/"ood_test.json","w") as fh:
    json.dump(out,fh,indent=2)
print()
print("saved",RESULTS/"ood_test.json")

RMSE, lower is better
                                dissolution   hardness      weight RSD    tensile
code 15 unseen, high cluster    2.895         18.69         0.364         0.143
code 25 unseen, control         3.299         4.918         0.587         0.303
code 15, fold 2 model           3.039         38.017        0.391         0.806
code 15, fold 3 model           2.84          32.633        0.353         0.419
cross validation average        3.261         7.698         0.585         0.345

PICP at 90 percent, 0.90 is the target
                                dissolution   hardness      weight RSD    tensile
code 15 unseen, high cluster    1.0           0.938         0.969         1.0
code 25 unseen, control         0.912         0.971         0.971         0.853
code 15, fold 2 model           1.0           0.391         1.0           0.922
code 15, fold 3 model           1.0           0.984         0.984         1.0

hardness, predicted against actual
  code 15 unseen, high